# Financial Distress & Bankruptcy Prediction

A machine learning project that predicts company bankruptcy using 64 anonymized financial attributes. The solution uses a **Stacked Ensemble** of three gradient boosting models (LightGBM, XGBoost, CatBoost) combined via a Logistic Regression meta-model, achieving an OOF AUC of **0.9056**.

**Author:** Harshil Panchal

---

**Approach:**
- **Base Models (Level 0):** LightGBM, XGBoost, CatBoost — each individually tuned via Bayesian optimization and trained with 5-fold Stratified K-Fold cross-validation to generate Out-of-Fold (OOF) predictions
- **Meta-Model (Level 1):** Logistic Regression (C=0.1) trained on OOF meta-features
- **Benefit:** Reduces individual model bias and leverages the unique strengths of each gradient booster, producing a more robust and accurate final prediction

## Model Architecture

The **Stacked Ensemble** combines predictions from three individually-tuned gradient boosting models:

| Level | Model | OOF AUC |
|-------|-------|---------|
| Base (L0) | LightGBM | 0.90420 |
| Base (L0) | XGBoost | 0.90379 |
| Base (L0) | CatBoost | 0.88780 |
| **Meta (L1)** | **Logistic Regression** | **0.90561** |

Each base model was trained using **5-fold Stratified K-Fold cross-validation** to generate Out-of-Fold (OOF) predictions, which serve as input features for the meta-model. This prevents data leakage and ensures the meta-model generalises well.

In [ ]:
# Install CatBoost if not already available
# !pip install catboost lightgbm xgboost scikit-learn pandas numpy matplotlib

# Uncomment below if running on Google Colab with Drive mounted:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Install required libraries (uncomment if needed)
# !pip install catboost lightgbm xgboost scikit-learn pandas numpy matplotlib

# --- Import necessary libraries ---
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA

# --- CONFIGURATION SETTINGS ---
RANDOM_STATE = 42
TARGET_COL = 'class'
ID_COL = 'ID'
N_SPLITS_STACKING = 5
EARLY_STOPPING_ROUNDS = 100

# --- 1. Data Loading ---
# Data files should be placed in the data/ directory relative to this notebook.
# If running on Google Colab with Drive, update the paths accordingly:
#   train_df = pd.read_csv("/content/drive/MyDrive/.../bankruptcy_Train.csv")
try:
    train_df = pd.read_csv("data/bankruptcy_Train.csv")
    test_df  = pd.read_csv("data/bankruptcy_Test_X.csv")
    print("Data files loaded successfully.")
except FileNotFoundError:
    raise FileNotFoundError(
        "Files not found. Please ensure 'bankruptcy_Train.csv' and "
        "'bankruptcy_Test_X.csv' are placed in the data/ directory."
    )

X_full   = train_df.drop(columns=[TARGET_COL])
y_full   = train_df[TARGET_COL]
X_test_id = test_df[ID_COL]
X_test   = test_df.drop(columns=[ID_COL])

# Ensure test columns match training columns — critical for consistent preprocessing.
X_test = X_test.reindex(columns=X_full.columns)
print("Data prepared for processing.")

# --- 2. Preprocessing Classes and Pipeline ---

class MissingIndicatorCreator(BaseEstimator, TransformerMixin):
    """Creates binary indicator columns for features that contain missing values."""
    def fit(self, X, y=None):
        self.na_cols = X.columns[X.isnull().any()].tolist()
        self.feature_names_in_ = X.columns.tolist()
        return self

    def transform(self, X, y=None):
        X_df = X.copy()
        for col in self.na_cols:
            if col in X_df.columns:
                X_df[f'{col}_isna'] = X_df[col].isnull().astype(int)
        return X_df

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        output_features = list(input_features)
        for col in self.na_cols:
            if col in input_features:
                output_features.append(f'{col}_isna')
        return np.asarray(output_features, dtype=object)


class AdvancedFeatureCreator(BaseEstimator, TransformerMixin):
    """Generates PCA components and pairwise interaction terms (ratios and products)."""
    def __init__(self, n_components=5):
        self.n_components = n_components
        self.pca = None
        # 1-indexed feature pairs for interaction terms
        self.interaction_pairs = [(2, 4), (10, 15), (25, 40), (50, 60), (3, 7)]

    def fit(self, X, y=None):
        self.feature_names_in_ = X.columns.tolist()
        self.pca = PCA(n_components=self.n_components)
        self.pca.fit(X)
        return self

    def transform(self, X, y=None):
        X_proc = X.copy()
        cols = X_proc.columns.tolist()
        for i, j in self.interaction_pairs:
            if len(cols) >= max(i, j) and i > 0 and j > 0:
                ci, cj = cols[i - 1], cols[j - 1]
                X_proc[f'Ratio_{ci}_{cj}']   = X_proc[ci] / (X_proc[cj] + 1e-6)
                X_proc[f'Product_{ci}_{cj}'] = X_proc[ci] * X_proc[cj]
        pca_df = pd.DataFrame(
            self.pca.transform(X),
            index=X_proc.index,
            columns=[f'PCA_{k+1}' for k in range(self.n_components)]
        )
        return pd.concat([X_proc, pca_df], axis=1)

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        output_features = list(input_features)
        for i, j in self.interaction_pairs:
            if len(input_features) >= max(i, j) and i > 0 and j > 0:
                ci, cj = input_features[i - 1], input_features[j - 1]
                output_features += [f'Ratio_{ci}_{cj}', f'Product_{ci}_{cj}']
        output_features += [f'PCA_{k+1}' for k in range(self.n_components)]
        return np.asarray(output_features, dtype=object)


PREPROCESSOR = Pipeline(steps=[
    ('indicator',      MissingIndicatorCreator()),
    ('imputer',        SimpleImputer(strategy='median').set_output(transform="pandas")),
    ('scaler',         RobustScaler().set_output(transform="pandas")),
    ('feature_creator', AdvancedFeatureCreator(n_components=5))
])
print("Preprocessing pipeline defined.")

# --- 3. Optimal Model Parameters (from Bayesian Optimization) ---
lgbm_final_params = {
    'colsample_bytree': 0.801212012041561,
    'learning_rate':    0.0857042821413322,
    'max_depth':        11,
    'n_estimators':     1250,
    'num_leaves':       25,
    'reg_alpha':        0.11754544962406674,
    'subsample':        0.8161984266286941,
    'n_jobs':           -1,
    'random_state':     RANDOM_STATE,
    'boosting_type':    'gbdt',
    'verbose':          -1,
    'objective':        'binary',
    'metric':           'auc'
}

xgb_final_params = {
    'colsample_bytree':  0.9732942780874417,
    'gamma':             0.16180837718610405,
    'learning_rate':     0.011377167581321596,
    'max_depth':         7,
    'min_child_weight':  5.0,
    'n_estimators':      1150,
    'reg_alpha':         0.18723139156591262,
    'subsample':         0.9233356022977088,
    'n_jobs':            -1,
    'random_state':      RANDOM_STATE,
    'objective':         'binary:logistic',
    'eval_metric':       'auc',
    'use_label_encoder': False,
    'tree_method':       'hist',
    'verbosity':         0
}

cat_final_params = {
    'colsample_bylevel': 0.8212992727681557,
    'depth':             8,
    'iterations':        1050,
    'l2_leaf_reg':       7.384908098277113,
    'learning_rate':     0.06357915054231,
    'random_seed':       RANDOM_STATE,
    'subsample':         0.722891298646186,
    'verbose':           0,
    'eval_metric':       'AUC',
    'objective':         'Logloss'
}
print("Model parameters loaded.")

# --- 4. Preprocessing ---
print("Preprocessing datasets...")
X_full_proc = PREPROCESSOR.fit_transform(X_full)
X_test_proc  = PREPROCESSOR.transform(X_test)
print("Preprocessing complete.")

# --- 5. Out-of-Fold Predictions (Stacking Base Layer) ---
print(f"Training base models with {N_SPLITS_STACKING}-fold StratifiedKFold...")

oof_preds_lgbm = np.zeros(X_full_proc.shape[0])
oof_preds_xgb  = np.zeros(X_full_proc.shape[0])
oof_preds_cat  = np.zeros(X_full_proc.shape[0])

test_preds_lgbm_folds, test_preds_xgb_folds, test_preds_cat_folds = [], [], []

skf = StratifiedKFold(n_splits=N_SPLITS_STACKING, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full_proc, y_full)):
    print(f"  Fold {fold+1}/{N_SPLITS_STACKING}")
    X_tr, X_val = X_full_proc.iloc[train_idx], X_full_proc.iloc[val_idx]
    y_tr, y_val = y_full.iloc[train_idx],      y_full.iloc[val_idx]

    # LightGBM
    lgbm_model = LGBMClassifier(**lgbm_final_params)
    lgbm_model.fit(X_tr, y_tr)
    oof_preds_lgbm[val_idx] = lgbm_model.predict_proba(X_val)[:, 1]
    test_preds_lgbm_folds.append(lgbm_model.predict_proba(X_test_proc)[:, 1])

    # XGBoost
    xgb_model = xgb.XGBClassifier(**xgb_final_params)
    xgb_model.fit(X_tr, y_tr)
    oof_preds_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    test_preds_xgb_folds.append(xgb_model.predict_proba(X_test_proc)[:, 1])

    # CatBoost
    cat_model = CatBoostClassifier(**cat_final_params)
    cat_model.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                  verbose=0)
    oof_preds_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]
    test_preds_cat_folds.append(cat_model.predict_proba(X_test_proc)[:, 1])

# Average test predictions across folds for stability
test_preds_lgbm = np.mean(test_preds_lgbm_folds, axis=0)
test_preds_xgb  = np.mean(test_preds_xgb_folds,  axis=0)
test_preds_cat  = np.mean(test_preds_cat_folds,   axis=0)

print(f"\nLightGBM OOF AUC : {roc_auc_score(y_full, oof_preds_lgbm):.5f}")
print(f"XGBoost  OOF AUC : {roc_auc_score(y_full, oof_preds_xgb):.5f}")
print(f"CatBoost OOF AUC : {roc_auc_score(y_full, oof_preds_cat):.5f}")

# --- 6. Meta-Model Training & Final Predictions ---
print("\nTraining meta-model (Logistic Regression)...")

meta_features_train = pd.DataFrame({'lgbm_pred': oof_preds_lgbm, 'xgb_pred': oof_preds_xgb, 'cat_pred': oof_preds_cat})
meta_features_test  = pd.DataFrame({'lgbm_pred': test_preds_lgbm,'xgb_pred': test_preds_xgb, 'cat_pred': test_preds_cat})

meta_model = LogisticRegression(solver='liblinear', random_state=RANDOM_STATE, C=0.1)
meta_model.fit(meta_features_train, y_full)

stacked_oof_auc = roc_auc_score(y_full, meta_model.predict_proba(meta_features_train)[:, 1])
print(f"Stacked Ensemble OOF AUC : {stacked_oof_auc:.5f}")

stacked_test_probas = meta_model.predict_proba(meta_features_test)[:, 1]

submission_df = pd.DataFrame({ID_COL: X_test_id, TARGET_COL: stacked_test_probas})
submission_df.to_csv('results/submission.csv', index=False)
print("\nSubmission saved to results/submission.csv")